# CardinalSplineBasis

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/basis/doc/CardinalSplineBasis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note: AI was used in the creation of this example.

[`CardinalSplineBasis`](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/CardinalSplineBasis.h) provides cubic cardinal-spline weights for interpolating scalar or vector coefficients at a known coordinate. This guide explains the weights and shows how the class fits GTSAM's basis-function API.

Use [`CumulativeSplineTrajectory<T>`](CumulativeSplineTrajectory.ipynb) instead when the controls are poses, rotations, or other Lie-group values.

Primary contributor: [Brett Downing](https://github.com/BrettRD).

GTSAM Copyright 2010-2022, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

## Contents

- [When to use it](#cardinal-spline-basis-when-to-use-it)
- [Basis weights](#cardinal-spline-basis-weights)
- [Python usage](#cardinal-spline-basis-python)
- [C++ usage](#cardinal-spline-basis-cpp)
- [Relationship to cumulative kernels](#cardinal-spline-basis-kernels)
- [Example and related class](#cardinal-spline-basis-related)
- [Source](#cardinal-spline-basis-source)

In [1]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not in Colab

In [2]:
import gtsam
import numpy as np

(cardinal-spline-basis-when-to-use-it)=
## When to use it

Use `CardinalSplineBasis` when:

- the coefficients are scalars or vectors;
- the sample coordinate is an ordinary numeric value; and
- you need cubic interpolation weights or a `Basis` evaluation functor.

Do not apply these weights directly to poses or rotations: scalar multiplication and addition are not the correct operations for Lie-group values. For those controls, use [`CumulativeSplineTrajectory<T>`](CumulativeSplineTrajectory.ipynb).

(cardinal-spline-basis-weights)=
## Basis weights

For coefficients $p_0, \ldots, p_{N-1}$, the interpolated value is the weighted sum

$$f(t)=\sum_{i=0}^{N-1} B_i(t)p_i.$$

`CalculateWeights` returns the dense vector $B(t)$. Only a small neighborhood has nonzero interior weights, while the first and last entries absorb the constant tails. The weights sum to one, so a constant set of coefficients remains constant.

(cardinal-spline-basis-python)=
## Python usage

Python exposes the class as `gtsam.CardinalSplineBasis`. `CalculateWeights(N, x)` returns the interpolation weights for `N` coefficients at the unit-spaced coordinate `x`; `DerivativeWeights(N, x)` returns the first-derivative weights. Both methods also accept `a` and `b` to map a bounded coordinate interval over the full spline support.

In [3]:
coefficients = np.array([1.0, 2.0, 0.5, 3.0])
weights = gtsam.CardinalSplineBasis.CalculateWeights(len(coefficients), 3.5)
derivative_weights = gtsam.CardinalSplineBasis.DerivativeWeights(
    len(coefficients), 3.5
)
value = weights @ coefficients
derivative = derivative_weights @ coefficients
np.testing.assert_allclose(weights.sum(), 1.0)
weights, value, derivative

(array([0.02083333, 0.47916667, 0.47916667, 0.02083333]),
 1.2812499999999996,
 -0.6875000000000002)

Taking the dot product of the returned weights with scalar coefficients evaluates the curve or its derivative. For vector coefficients, arrange the values as columns and apply the same weights along the coefficient axis.

(cardinal-spline-basis-cpp)=
## C++ usage

The class can provide a functor to GTSAM's generic basis evaluation machinery:

```cpp
Vector coefficients{1.0, 2.0, 0.5, 3.0};
CardinalSplineBasis::EvaluationFunctor evaluate(coefficients.size(), 3.5);
double value = evaluate(coefficients);
```

`EvaluationFunctor` stores the weights for the chosen coordinate. `DerivativeFunctor` does the same for a derivative order. These functors can also be used with the generic evaluation factors declared in `Basis.h`.

(cardinal-spline-basis-kernels)=
## Relationship to cumulative kernels

The implementation obtains the dense weights from shifted cumulative-kernel activations $c_i(t)$. For $N$ coefficients,

$$B_0=1-c_1, \qquad B_i=c_i-c_{i+1}, \qquad B_{N-1}=c_{N-1}.$$

This difference-of-cumulative-steps form explains why the weights are local and sum to one. For scalar or vector coefficients, it is algebraically equivalent to starting at $p_0$ and cumulatively adding weighted differences $p_i-p_{i-1}$.

That equivalence does **not** make the two GTSAM classes interchangeable. `CardinalSplineBasis` forms an ordinary linear combination of coefficients. [`CumulativeSplineTrajectory<T>`](CumulativeSplineTrajectory.ipynb) maps relative pose or rotation changes through the Lie-group logarithm and exponential, and it supports expression-valued time and bounded windows.

(cardinal-spline-basis-related)=
## Example and related class

- [CardinalSplineBasis scalar example](../../../python/gtsam/examples/CardinalSplineBasisExample.ipynb) plots the weights, interpolated curve, and derivatives.
- [CumulativeSplineTrajectory](CumulativeSplineTrajectory.ipynb) covers pose and rotation trajectories.

(cardinal-spline-basis-source)=
## Source

- [`CardinalSplineBasis.h`](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/CardinalSplineBasis.h)
- [`Basis.h`](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/Basis.h)
- [`IrwinHall.h`](https://github.com/borglab/gtsam/blob/develop/gtsam/basis/IrwinHall.h)